In [1]:
# Main to test model parameter influence

In [2]:
# Check this before running a new test
# 1. Select the right model including or excluding the nucleus area fraction
# 2. uncomment the correct line in the train and test functions that does or does not used the area fraction
# 3. select the right type for the ablation study
# 4. uncomment the linear regression model if area fraction is included

In [3]:
type = 'nucleusSI'
RandSeedNumber = 89523
nr = '001'

In [4]:
import os
import numpy as np
import torch
import torch_geometric
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader 
from GNNmodel_Model_BatchNorm_EdgeAttr_noAF import myPNA
from GNNmodel_Visuals import visualize_snapshot_cell, visualize_snapshot_nucleus, visualize_snapshot_graph, lin_regression
import scipy.stats
import seaborn as sns
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy import io
import pickle
import random
import copy
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
torch.cuda.empty_cache()
plt.rcParams['font.family'] = 'serif'
print(torch.cuda.is_available())

plt.rcParams['xtick.direction'] = plt.rcParams['ytick.direction'] = 'in'

/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /vast.mnt/home/20234017/.local/lib/python3.11/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  import torch_geometric.typing


True


In [5]:
# function for loading the processed data

def load_dataset(file_name):
    with open(file_name, 'rb') as f:
        data = pickle.load(f)
        return data

In [6]:
# load data
data_list_all = load_dataset('AllProcessedData_BigGrid_batch3_[FINAL].pkl')
data_list_validation = load_dataset('AllProcessedData_BigGrid_batch3_[FINAL_VALIDATION].pkl')

In [7]:
# Reduce amount of low D values by only taking 2 simulations from each of the specified setting pairs

selected_indices = {
    0, 1, 2,
    9, 10, 11,
    18, 19, 20,
    27, 28, 29,
    36, 37, 38,
    45, 46, 47,
    54, 55, 56, 57, 58,
    63, 64, 65, 66, 67, 68,
    72, 73, 74, 75, 76, 77, 78, 79}

filtered_data_list = []

for block_idx in range(81):
    start = block_idx * 12
    end = start + 12

    block = data_list_all[start:end]

    if block_idx in selected_indices:
        filtered_data_list.extend(copy.deepcopy(block[10:]))
    else:
        filtered_data_list.extend(copy.deepcopy(block))

print(f"Original length: {len(data_list_all)}")
print(f"Filtered length: {len(filtered_data_list)}")

Original length: 972
Filtered length: 602


In [8]:
# function for setting random seed number

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # GPU will be slower but deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 
    print("Seeded everything: {}".format(seed))

In [9]:
# Get the statistics to normalize the input

def compute_stats(data_list):
    cell_area = []
    cell_perim = []
    cell_si = []
    nuc_area = []
    nuc_perim = []
    nuc_si = []
    diff = []
    edge_attr = []
    x_pos = []
    y_pos = []

    for data in data_list:
        cell_area.extend(data.x[:, 3])
        cell_perim.extend(data.x[:, 4])
        cell_si.extend(data.x[:, 5])
        nuc_area.extend(data.x[:, 0])
        nuc_perim.extend(data.x[:, 1])
        nuc_si.extend(data.x[:, 2])
        diff.extend([data.diffConst.item()])
        edge_attr.extend(data.edge_attr)
        x_pos.extend(data.cell_pos[:, 0])
        y_pos.extend(data.cell_pos[:, 1])

    stats = {
        "cell_area": (np.mean(cell_area), np.std(cell_area)),
        "cell_perim": (np.mean(cell_perim), np.std(cell_perim)),
        "cell_si": (np.mean(cell_si), np.std(cell_si)),
        "nuc_area": (np.mean(nuc_area), np.std(nuc_area)),
        "nuc_perim": (np.mean(nuc_perim), np.std(nuc_perim)),
        "nuc_si": (np.mean(nuc_si), np.std(nuc_si)),
        "diff": (np.mean(diff), np.std(diff)),
        "edge_attr": (np.mean(edge_attr), np.std(edge_attr)),
        "x_pos": (np.mean(x_pos), np.std(x_pos)),
        "y_pos": (np.mean(y_pos), np.std(y_pos)),
    }
    return stats

In [10]:
# function for picking diffusion constant for given snapshot as output

def setOutput(data_list):
    for data in data_list:
        data.y = data.diffConst
    return data_list

In [11]:
# function for choosing the node and edge information as the input

def setInput(data_list, type):
    for data in data_list: 
        if type == 'independent':
            data.x = data.x[:, [2,3,5]]  
        elif type == 'SI':
            data.x = data.x[:, [2,5]]  
        elif type == 'SI_noAF':
            data.x = data.x[:, [2,5]] # REMOVE PHI
        elif type == 'nucleusSI':
            data.x = data.x[:, [2]] # REMOVE PHI
        elif type == 'cellSI':
            data.x = data.x[:, [5]]  # REMOVE PHI
        elif type == 'nuclear':
            data.x = data.x[:, [0,1,2]]
        elif type == 'nuclear_areaPerim':
            data.x = data.x[:, [0,1]]  # REMOVE PHI
        elif type == 'nuclear_noAF':
            data.x = data.x[:, [0,1,2]]  # REMOVE PHI
        elif type == 'cellular':
            data.x = data.x[:, [3,4,5]]  # REMOVE PHI
        elif type == 'all':
            data.x = data.x
    return data_list

In [12]:
# Function to normalize all input data 

def normalize(data_list, stats):
    for data in data_list:
        
        # x features
        data.x[:, 3] = (data.x[:, 3] - stats["cell_area"][0]) / stats["cell_area"][1]
        data.x[:, 4] = (data.x[:, 4] - stats["cell_perim"][0]) / stats["cell_perim"][1]
        data.x[:, 5] = (data.x[:, 5] - stats["cell_si"][0]) / stats["cell_si"][1]

        data.x[:, 0] = (data.x[:, 0] - stats["nuc_area"][0]) / stats["nuc_area"][1]
        data.x[:, 1] = (data.x[:, 1] - stats["nuc_perim"][0]) / stats["nuc_perim"][1]
        data.x[:, 2] = (data.x[:, 2] - stats["nuc_si"][0]) / stats["nuc_si"][1]

        # other features
        data.diffConst = data.diffConst / stats["diff"][1]
        data.edge_attr = (data.edge_attr - stats["edge_attr"][0]) / stats["edge_attr"][1]
        data.cell_pos[:,0] = (data.cell_pos[:,0] - stats["x_pos"][0]) / stats["x_pos"][1]
        data.cell_pos[:,1] = (data.cell_pos[:,1] - stats["y_pos"][0]) / stats["y_pos"][1]

    return data_list

In [13]:
# Function to train the model and input area fraction as graph level

def train():
    model.train()
    for data in train_loader:
        optimizer.zero_grad()

        edge_attr = data.edge_attr.view(-1, 1)
        
        # calculate loss
        #pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda(), data.area_frac.cuda())
        pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda()) # FOR NO PHI
        loss = criterion(pred.squeeze(), data.y.squeeze().cuda())

        # update
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    return pred.squeeze().detach().cpu(), data.y.squeeze()

In [14]:
@torch.no_grad()
def test(loader):
    model.eval()
    loss_list=[]
    pred_list=[]
    y_list=[]
    for data in loader:
        edge_attr = data.edge_attr.view(-1, 1)
        #pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda(), data.area_frac.cuda())
        pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda()) # FOR NO PHI
        loss = criterion(pred.squeeze().detach().cpu(), data.y.squeeze())
        loss_list.append(loss.item())
        pred_list.extend(pred.squeeze().detach().cpu())
        y_list.extend(data.y.squeeze())
    return loss_list, pred_list, y_list

In [15]:
# Function to randomly select test and train indices from the 30 simulations out 300

def select_indices():
 
    selected_idx = np.random.choice(np.arange(len(filtered_data_list)), size=len(filtered_data_list), replace=False)
    #test_idx = np.random.choice(selected_idx, size=int(0.2*len(selected_idx)), replace=False)
    #np.save("TrainingResults_batch3_gridBig_[FeatureAblation]/test_idx_FA.npy", test_idx)
    test_idx = np.load("TrainingResults_batch3_gridBig_[FeatureAblation]/test_idx_FA.npy")

    remaining_idx = np.setdiff1d(selected_idx, test_idx)
    train_idx = np.random.choice(remaining_idx, size=int(0.6*len(selected_idx)), replace=False)

    val_idx = np.random.choice(np.arange(972), size=972, replace=False)

    test_idx_list = test_idx.tolist()
    train_idx_list = train_idx.tolist()
    val_idx_list = val_idx.tolist()

    return test_idx_list, train_idx_list, val_idx_list

In [16]:
# MAIN CODE WHERE TRAINING ACTUALLY HAPPENS

In [17]:
# create folders for saving results
os.makedirs('TrainingResults_batch3_gridBig_[FeatureAblation]', exist_ok=True)
os.makedirs('TrainingWeights_batch3_gridBig_[FeatureAblation]', exist_ok=True)

In [18]:
# rand seed initialization
set_seed(RandSeedNumber)

Seeded everything: 89523


In [19]:
# Select the indices of the simulations used for training and testing

test_idx, train_idx, val_idx = select_indices()

In [20]:
# Create the list with the actual data to train and test with

data_list_train = []
data_list_test = []
data_list_val = []

for i, item in enumerate(filtered_data_list):  # 602 simulations
    if i in test_idx:
        data_list_test.extend(item)    

for i, item in enumerate(filtered_data_list):
    if i in train_idx:
        data_list_train.extend(item)

for i, item in enumerate(data_list_validation):  # 972 simulations
    if i in val_idx:
        data_list_val.extend(item) 

print(f'Train size:{len(data_list_train)}, Test size:{len(data_list_test)}, Validation size:{len(data_list_val)}')

Train size:1805, Test size:600, Validation size:2916


In [21]:
# Compute mean and std for all features

stats = compute_stats(data_list_train)

In [22]:
# Obtain a histogram of the initial diffusion consyant distribution of the test dataset

D_init = []
for data in data_list_test:
    D_init.append(data.diffConst)

plt.hist(D_init, bins=10, edgecolor='black')
plt.xlabel("D")
plt.ylabel("Counts")
plt.title("Initial D")

plt.savefig("TrainingResults_batch3_gridBig_[FeatureAblation]/DinitTest_001.png", dpi=200, bbox_inches="tight")
plt.close()

In [24]:
# nondimesionalize lists

data_train_nondim = normalize(data_list_train, stats)
data_test_nondim = normalize(data_list_test, stats)
data_val_nondim = normalize(data_list_val, stats)

In [28]:
# Shuffle the content of both lists since now all snapshots from same simulation are clustered together

random.shuffle(data_train_nondim)
random.shuffle(data_test_nondim)
random.shuffle(data_val_nondim)

In [29]:
# Select the diffusion constant as the output

data_train_nondim = setOutput(data_train_nondim)
data_test_nondim = setOutput(data_test_nondim)
data_val_nondim = setOutput(data_val_nondim)

In [30]:
# Select the node features and edges as the input 

data_train_nondim = setInput(data_train_nondim, type)
data_test_nondim = setInput(data_test_nondim, type)
data_val_nondim = setInput(data_val_nondim, type) 

In [31]:
# Visualize snapshots and graph representation of a certain snapshot

visualize_snapshot_cell(data_val_nondim, [50, 100])
visualize_snapshot_nucleus(data_val_nondim, [50, 100])
visualize_snapshot_graph(data_val_nondim, [50, 100])

In [32]:
# Settings for the model - MAIN VARIABLES

num_layers      = 5     
hidden_channels = 24     
batch_size      = 64   
learning_rate   = 1e-5   
weight_decay    = 1e-3   
EpochNum        = 201 

In [33]:
# Test what the capabilities of a linear regression model are on phi

# corr, MSE, p_value = lin_regression(data_list_val)
# print(f'The correlation is: {corr}')
# print(f'The MSE is: {MSE}') 
# print(f'The p-value is: {p_value}')

In [34]:
g = torch.Generator()
g.manual_seed(RandSeedNumber)

In [35]:
# Initialize Dataloader to make data ready to train model with

train_loader = DataLoader(data_train_nondim, batch_size=batch_size, shuffle=True, generator=g, drop_last=True)
test_loader = DataLoader(data_test_nondim, batch_size=4096, shuffle=False, generator=g, drop_last=False) 

In [36]:
# initialize model

in_channels = data_list_train[0].x.shape[1] # Shape has (num_nodes, features) and you only want features
model = myPNA(data_list=data_train_nondim, in_channels=in_channels, hidden_channels=hidden_channels, num_layers=num_layers).cuda()
print(model)

tensor([    0,   390,  4930, 19234, 25520, 34096, 79757, 16075,   497,     1])
myPNA(
  (convs): ModuleList(
    (0): PNAConv(1, 24, towers=3, edge_dim=1)
    (1-4): 4 x PNAConv(24, 24, towers=3, edge_dim=1)
  )
  (batch_norms): ModuleList(
    (0-4): 5 x BatchNorm(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (lin): Linear(in_features=24, out_features=1, bias=True)
)


In [37]:
# initialize optimizer

criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp) 

In [39]:
# train model

train_loss_lst=[]
test_loss_lst=[]
train_corr_lst=[]
test_corr_lst=[]
for epoch in range(1, EpochNum):
    pred, truth = train()

    with torch.no_grad():
        train_loss, train_pred, train_truth = test(train_loader)
        test_loss, test_pred, test_truth = test(test_loader)
        train_loss_lst.append(sum(train_loss)/len(train_loss))
        test_loss_lst.append(sum(test_loss)/len(test_loss))

        # calculate correlation
        train_corr, train_p_value = scipy.stats.pearsonr(np.array(train_pred),np.array(train_truth))
        test_corr, test_p_value = scipy.stats.pearsonr(np.array(test_pred),np.array(test_truth))

        train_corr_lst.append(train_corr)
        test_corr_lst.append(test_corr)

        print(f'Epoch: {epoch:03d}, Train loss: {sum(train_loss)/len(train_loss):.4f}, Test loss: {sum(test_loss)/len(test_loss):.4f},\
            Pearson correlation (train): {train_corr:.4f}, Pearson correlation (test): {test_corr:.4f}')

/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch: 001, Train loss: 0.9362, Test loss: 0.8415,            Pearson correlation (train): 0.6766, Pearson correlation (test): 0.6637
Epoch: 002, Train loss: 0.7881, Test loss: 0.7275,            Pearson correlation (train): 0.6684, Pearson correlation (test): 0.6392
Epoch: 003, Train loss: 0.7027, Test loss: 0.6515,            Pearson correlation (train): 0.6908, Pearson correlation (test): 0.6618
Epoch: 004, Train loss: 0.6466, Test loss: 0.5970,            Pearson correlation (train): 0.7226, Pearson correlation (test): 0.6945
Epoch: 005, Train loss: 0.5953, Test loss: 0.5474,            Pearson correlation (train): 0.7494, Pearson correlation (test): 0.7237
Epoch: 006, Train loss: 0.5540, Test loss: 0.5075,            Pearson correlation (train): 0.7699, Pearson correlation (test): 0.7475
Epoch: 007, Train loss: 0.5131, Test loss: 0.4763,            Pearson correlation (train): 0.7849, Pearson correlation (test): 0.7640
Epoch: 008, Train loss: 0.4741, Test loss: 0.4454,            

In [40]:
# POST PROCESSING AND VISUALIZATION

In [41]:
# Plot the loss curves

def plot_losses(train, test):
    plt.figure()
    plt.semilogy(train, label="train", marker=".", lw=2)
    plt.semilogy(test, label="test", marker=".", lw=2)
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.savefig(f"TrainingResults_batch3_gridBig_[FeatureAblation]/losses_[{type}]_{RandSeedNumber}_{nr}_noSPstand.png", dpi=200)
    plt.close()

plot_losses(train_loss_lst, test_loss_lst)

In [42]:
# Test the model on the separate set of validation data 

val_loader = DataLoader(data_val_nondim, batch_size=2920, shuffle=False, generator=g, drop_last=False) 
val_loss, val_pred, val_truth = test(val_loader)

In [43]:
# Scale the model outputs back according to the pre-processing

truth = np.array(val_truth) * stats["diff"][1] + stats["diff"][0]
pred  = np.array(val_pred) * stats["diff"][1] + stats["diff"][0]

In [44]:
print(pred.min())
print(truth.min())

-0.4961146
0.011749148


In [45]:
# Obtain Pearson correlation of the validation dataset after scaling back

val_corr, val_p_value = scipy.stats.pearsonr(np.array(pred),np.array(truth))
print(f'The Pearson correlation is: {val_corr}')
print(f'The p-value is: {val_p_value}')

The Pearson correlation is: 0.8965762924751332
The p-value is: 0.0


In [47]:
# obtain least-squares fit parameters

m, b = np.polyfit(truth, pred, 1)
print(f'The slope is: {m} and the intercept is: {b}')

The slope is: 0.8077188005002529 and the intercept is: 0.2997959372269343


In [51]:
# 2D probability density scatter plot of predictions

plt.figure(figsize=(4, 3))
ax = sns.kdeplot(x=np.array(truth), y=np.array(pred), cmap="rocket_r", fill=True, levels=10, cbar=True, cut=12)
plt.scatter(np.array(truth),np.array(pred), s=5, c='lightblue', alpha=0.6, label = 'Data')

# reference line
xref = np.linspace(0, 6.0, num=100)
plt.plot(xref,xref, c = 'k', label = 'Reference',linewidth = 1, linestyle = 'dashed')

# plot line with intercept
x = np.linspace(0, 6, 100)
y2 = m*x + b
plt.plot(x,y2, label = "Line", linewidth = 1)

plt.xlim([0,6.0])
plt.ylim([0,6.0])
plt.xlabel('$100D$',fontsize=12)
plt.ylabel('$100D_{NN}$',fontsize=12)
plt.tight_layout()
plt.savefig(f'TrainingResults_batch3_gridBig_[FeatureAblation]/Fit_Line_noLog_[{type}]_{val_corr:.3f}_{RandSeedNumber}_{nr}_noSPstand.png', dpi=300, bbox_inches='tight')
plt.close()

In [49]:
# save results

savePath = f'TrainingResults_batch3_gridBig_[FeatureAblation]/GNNmodel_batch3GE_[{type}]_{nr}_noSPstand_layers{num_layers}_channels{hidden_channels}_batchSize{batch_size}_lr{learning_rate}_Epochs{EpochNum}_WeightDecay{weight_decay}_seed{RandSeedNumber}.mat'
io.savemat(savePath, {'train_loss': train_loss_lst,  'train_corr': train_corr_lst,
                      'test_loss' : test_loss_lst,   'test_corr': test_corr_lst})
print(savePath)

TrainingResults_batch3_gridBig_[FeatureAblation]/GNNmodel_batch3GE_[nucleusSI]_001_noSPstand_layers5_channels24_batchSize64_lr1e-05_Epochs201_WeightDecay0.001_seed89523.mat


In [50]:
# save trained model weights

torch.save(model.state_dict(), f'TrainingWeights_batch3_gridBig_[FeatureAblation]/GNNmodel_weights_[{type}]_{RandSeedNumber}_{nr}_noSPstand.pth')

In [51]:
# Plot the average Pearson correlations with their SEM

x_labels = ["2,3,5,phi", "2,5,phi", "2,5", "2", "5", "0,1,2,phi", "0,1,2", "0,1", "3,4,5", "0,1,2,3,4,5,phi"]
x = np.arange(len(x_labels))

values = np.array([0.9285, 0.9303, 0.9268, 0.8981, 0.9367, 0.8947, 0.9010, 0.9013, 0.9322, 0.9308])
std    = np.array([0.0024, 0.0013, 0.0015, 0.0022, 0.0011, 0.0023, 0.0045, 0.0042, 0.0012, 0.0017])

plt.figure(figsize=(5, 4))
ax = plt.gca()

plt.errorbar(x, values, yerr=std, fmt='o', capsize=4)

# ticks
plt.xticks(x, x_labels, rotation=30, ha='right')
ax.minorticks_on()
ax.tick_params(axis='x', which='minor', bottom=False)

# labels
plt.ylabel("Pearson correlation")
plt.title("Feature ablation study")

# grid (horizontal only)
ax.yaxis.grid(True, which='major', linewidth=0.5, alpha=0.3)
ax.yaxis.grid(True, which='minor', linewidth=0.3, alpha=0.15)
ax.xaxis.grid(False)

plt.tight_layout()

plt.savefig(
    "TrainingResults_batch3_gridBig_[FeatureAblation]/FeatureAblationResult_5seeds_report004.png",
    dpi=300,
    bbox_inches="tight")

plt.close()

In [4]:
# ANOVA tests to see significance of results 

# Pearson data
g1 = [0.929936667, 0.933766673, 0.929351273, 0.929991481, 0.919409495]
g2 = [0.932634178, 0.926362056, 0.932482691, 0.931835677, 0.928392185]
g3 = [0.926151779, 0.923009262, 0.931660634, 0.924644508, 0.928709418]
g4 = [0.898105265, 0.904135337, 0.900158706, 0.890598552, 0.897362112]
g5 = [0.937672956, 0.933018832, 0.936013425, 0.939674825, 0.937334139]
g6 = [0.898928208, 0.887792428, 0.90044096, 0.891950409, 0.894504681]
g7 = [0.905583715, 0.903332017, 0.88342893, 0.904017198, 0.908849005]
g8 = [0.891195414, 0.909478598, 0.904184611, 0.89166, 0.91016985]
g9 = [0.934550661, 0.934559592, 0.932904206, 0.930780272, 0.928084023]
g10 = [0.932886761, 0.932466894, 0.925666491, 0.928101405, 0.935032529]


# ANOVA
F, p = f_oneway(g1, g2, g3, g4, g5, g6, g7, g8, g9, g10)

print("ANOVA:")
print(f"F = {F:.4f}")
print(f"p = {p:.4e}")

# Tukey HSD
data = g1 + g2 + g3 + g4 + g5 + g6 + g7 + g8 + g9 + g10

labels = (
    ["2,3,5,phi"] * 5 +
    ["2,5,phi"] * 5 +
    ["2,5"] * 5 +
    ["2"] * 5 +
    ["5"] * 5 +
    ["0,1,2,phi"] * 5 +
    ["0,1,2"] * 5 +
    ["0,1"] * 5 +
    ["3,4,5"] * 5 +
    ["0,1,2,3,4,5,phi"] * 5)

tukey = pairwise_tukeyhsd(endog=data, groups=labels, alpha=0.05)

print("\nTukey HSD:")
print(tukey)

ANOVA:
F = 44.9727
p = 3.9020e-18

Tukey HSD:
         Multiple Comparison of Means - Tukey HSD, FWER=0.05          
     group1          group2     meandiff p-adj   lower   upper  reject
----------------------------------------------------------------------
            0,1           0,1,2  -0.0003    1.0 -0.0122  0.0116  False
            0,1 0,1,2,3,4,5,phi   0.0295    0.0  0.0176  0.0414   True
            0,1       0,1,2,phi  -0.0066 0.6948 -0.0185  0.0053  False
            0,1               2  -0.0033 0.9949 -0.0152  0.0086  False
            0,1       2,3,5,phi   0.0272    0.0  0.0152  0.0391   True
            0,1             2,5   0.0255    0.0  0.0136  0.0374   True
            0,1         2,5,phi    0.029    0.0  0.0171  0.0409   True
            0,1           3,4,5   0.0308    0.0  0.0189  0.0427   True
            0,1               5   0.0354    0.0  0.0235  0.0473   True
          0,1,2 0,1,2,3,4,5,phi   0.0298    0.0  0.0179  0.0417   True
          0,1,2       0,1,2,phi